# First training session on the 32k corpus

Trains `frs-s`-shaped weights (hidden 768, 8 layers, 32k vocabulary) on the real
32k corpus on one T4.

Two things this notebook exists to measure, because neither is known:

1. **step time for a real model** — the toy probe ran at 1.6 s/step at hidden 64;
   a 768-wide model with a 32k head is a different order;
2. **whether the loss falls on real text** — the synthetic corpus proved the
   trainer, not the data.

It starts with a bounded slice so the first session produces a number quickly.


In [ ]:
import os, subprocess
smi = subprocess.run(['bash','-lc','nvidia-smi -L'], capture_output=True, text=True).stdout
print(smi)
assert 'T4' in smi, 'no T4 in this session'

SMOKE = os.environ.get('SMOKE', '1') == '1'
# 2 MB of text is ~130 steps at batch 4 x 512: enough to see a trend and a step time.
TEXT_BYTES = 2_000_000 if SMOKE else 200_000_000
STEPS_LOG_EVERY = 5
print('smoke', SMOKE, 'text bytes', TEXT_BYTES)


In [ ]:
import glob, json, os, re
from pathlib import Path

def run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

# graphics userspace, forced: libnvidia-gl-580-server ships 15_nvidia_gbm.json,
# which libnvidia-egl-gbm1 already owns, so dpkg aborts without the overwrite.
r = run(['bash','-lc','apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq '
        '-o Dpkg::Options::="--force-overwrite" -o Dpkg::Options::="--force-confold" libnvidia-gl-580-server 2>&1 | tail -3'])
print('graphics userspace:', r.returncode)
assert Path('/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0').exists(), 'no libGLX_nvidia.so'

debs = sorted(Path('/kaggle/input').rglob('ferrisres_0.*_amd64.deb'))
assert debs, 'attach sectionme/ferrisres'
deb = debs[-1]
print('deb:', deb)
print(run(['dpkg','-i',str(deb)]).stdout[-600:])
ver = run(['bash','-lc','ferrisres --version']).stdout.strip()
print(ver)
assert '0.2.7' in ver or '0.2.8' in ver or '0.2.9' in ver, f'need a build with --tokenizer on train, got {ver}'

BIN = run(['bash','-lc','command -v ferrisres']).stdout.strip()
icd = sorted(glob.glob('/usr/share/vulkan/icd.d/*.json'))
os.environ['VK_DRIVER_FILES'] = icd[0]
os.environ['VK_ICD_FILENAMES'] = icd[0]
os.environ['RUST_LOG'] = 'info'
probe = run([BIN,'devices','--json'], env={**os.environ})
clean = re.sub(r'\x1b\[[0-9;]*m','',probe.stdout)
i = clean.rfind('\n[')
devs = json.loads(clean[i+1:] if i >= 0 else clean[clean.find('['):])
t4 = [d for d in devs if 'T4' in d.get('name','') and not d.get('software')]
assert t4, f'no hardware T4: {devs}'
os.environ['WGPU_ADAPTER_INDEX'] = str(t4[0]['index'])
print('adapter', t4[0]['index'], t4[0]['name'])


In [ ]:
# train consumes text, and the corpus holds 32k BPE ids, so decode back to text
# with the same tokenizer. Byte-level BPE is lossless, so the trainer re-encodes
# to the same ids.
import struct
from tokenizers import Tokenizer

TOK = sorted(Path('/kaggle/input').rglob('tokenizer.json'))
SHARDS = sorted(Path('/kaggle/input').rglob('train-*.bin'))
assert TOK and SHARDS, 'attach sectionme/ferrisres-corpus-32k'
tok = Tokenizer.from_file(str(TOK[0]))
print('tokenizer', TOK[0], 'vocab', tok.get_vocab_size())

raw = SHARDS[0].read_bytes()
ids = list(struct.unpack('<%dH' % (len(raw)//2), raw))
text = tok.decode(ids, skip_special_tokens=False)
print('ids', len(ids), 'chars', len(text))
text = text[:TEXT_BYTES]
DATA = Path('/kaggle/working/train.txt'); DATA.write_text(text)
print('wrote', DATA, DATA.stat().st_size, 'bytes')
assert len(text) > 100_000, 'slice too small to train on'


In [ ]:
import re
import time

CKPT = Path('/kaggle/working/frs-s.ckpt')
cmd = [BIN, 'train',
       '--data', str(DATA), '--data-format', 'text',
       '--tokenizer', str(TOK[0]),
       '--hidden-dim', '768', '--num-blocks', '2', '--block-size', '4',
       '--seq-len', '512', '--batch-size', '4', '--learning-rate', '1e-3',
       '--epochs', '1', '--seed', '42',
       '--checkpoint-every', '50', '--output', str(CKPT),
       '--log-every', str(STEPS_LOG_EVERY),
       '--device', 'gpu']
print(' '.join(cmd))

t0 = time.perf_counter()
proc = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ})
elapsed = time.perf_counter() - t0
log = re.sub(r'\x1b\[[0-9;]*m', '', proc.stdout + proc.stderr)
print(log[-6000:])
Path('/kaggle/working/train.log').write_text(log)

import re
m = re.search(r'steps=(\d+).*initial_loss=([0-9.]+) final_loss=([0-9.]+)', log)
print('exit', proc.returncode, 'elapsed', round(elapsed, 1), 's')
if m:
    steps, init, final = int(m.group(1)), float(m.group(2)), float(m.group(3))
    print(f'steps={steps} initial={init:.6f} final={final:.6f} '
          f'step_time={elapsed/steps:.3f}s tokens_per_s={steps*4*512/elapsed:.0f}')
    Path('/kaggle/working/session.json').write_text(json.dumps({
        'steps': steps, 'initial_loss': init, 'final_loss': final,
        'seconds': round(elapsed, 2), 'step_time_s': round(elapsed/steps, 3),
        'tokens_per_s': round(steps*4*512/elapsed, 1),
        'vocab': tok.get_vocab_size(), 'smoke': SMOKE,
    }, indent=2))
else:
    print('no training summary found; the log above is the evidence')
